# 02 Xci Analysis

xCI analysis notebook (Python)

Purpose: compute within-group and cross-group concordance metrics for PREVENT / PCE analyses.

# Initialize Truveta SDK

In [ ]:
from truveta.study import Client, OutputMode, display_df
import pyspark.pandas as ps
import pandas as pd
import numpy as np

In [ ]:
# Use only one statement below and comment out whichever you are not using.
client = Client(output_mode = OutputMode.PandasOnSpark)
# client = Client(output_mode = OutputMode.PySpark)

In [ ]:
study = client.get_study()
# Use only one statement below and comment out whichever you are not using.
# population = study.get_population(title = "PREVENT and PCE")
population = study.get_population(id = "<TRUVETA_POPULATION_ID>")
population

In [ ]:
# Get latest completed active snapshot.
snapshot = population.get_latest_snapshot()
snapshot

In [ ]:
# Show tables in the snapshot.
snapshot.get_tables()

# Prepare data

In [ ]:
df = study.load_artifacts_data("/PREVENT_PCE/Aim_1/original_prevent_result")
display_df(df)
df = df.to_pandas()

## Addressing comments on EGFR

In [ ]:
mean_egfr = df['egfr'].mean()
var_egfr = df['egfr'].std()
group_stats = df.groupby('ascvd')['egfr'].agg(['mean', 'std'])
print(mean_egfr)
print(var_egfr)
print(group_stats)

# Calculate xCI

In [ ]:
def xCI_spark(s_test, t_test, pred_risk,
              weights=None,
              pos_group=None, neg_group=None,
              return_num_valid=False,
              tied_tol=1e-8):
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, when, abs as abs_
    from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, BooleanType
    
    # Initialize SparkSession if not already available
    spark = SparkSession.builder.getOrCreate()
    
    # Convert NumPy arrays to Python lists
    s_test_list = s_test.astype(int).tolist()
    t_test_list = t_test.astype(float).tolist()
    pred_risk_list = pred_risk.astype(float).tolist()
    
    # Handle weights if provided
    if weights is not None:
        weights_list = weights.astype(float).tolist()
    else:
        weights_list = [1.0] * len(s_test_list)
    
    # Convert pos_group and neg_group if provided
    if pos_group is not None:
        pos_group_list = pos_group.astype(bool).tolist()
    else:
        pos_group_list = [True] * len(s_test_list)
    
    if neg_group is not None:
        neg_group_list = neg_group.astype(bool).tolist()
    else:
        neg_group_list = [True] * len(s_test_list)
    
    # Create a list of tuples representing each row
    data = list(zip(s_test_list, t_test_list, pred_risk_list,
                    weights_list, pos_group_list, neg_group_list))
    
    # Define the schema explicitly
    schema = StructType([
        StructField('id', IntegerType(), False),
        StructField('s_test', IntegerType(), True),
        StructField('t_test', FloatType(), True),
        StructField('pred_risk', FloatType(), True),
        StructField('weights', FloatType(), True),
        StructField('pos_group', BooleanType(), True),
        StructField('neg_group', BooleanType(), True)
    ])
    
    # Add unique IDs to each row for joining purposes
    data_with_id = [(i,) + row for i, row in enumerate(data)]
    
    # Create the Spark DataFrame
    df = spark.createDataFrame(data_with_id, schema=schema)
    
    # Prepare the positive group DataFrame
    df_pos = df.filter((col('s_test') == 1) & (col('pos_group') == True))
    df_pos = df_pos.select(col('id').alias('id_pos'),
                           col('t_test').alias('t_test_pos'),
                           col('pred_risk').alias('pred_risk_pos'),
                           col('weights').alias('weights_pos'))
    
    # Prepare the negative group DataFrame
    df_neg = df.filter(col('neg_group') == True)
    df_neg = df_neg.select(col('id').alias('id_neg'),
                           col('t_test').alias('t_test_neg'),
                           col('pred_risk').alias('pred_risk_neg'),
                           col('weights').alias('weights_neg'))
    
    # Perform cross join between positive and negative groups
    df_pairs = df_pos.crossJoin(df_neg)
    
    # Apply the time condition
    df_valid = df_pairs.filter(col('t_test_pos') < col('t_test_neg'))
    
    # Compute weight product
    df_valid = df_valid.withColumn('weight_product', col('weights_pos') * col('weights_neg'))
    
    # Compute risk differences
    df_valid = df_valid.withColumn('risk_diff', col('pred_risk_pos') - col('pred_risk_neg'))
    
    # Determine correctly ranked and tied
    df_valid = df_valid.withColumn(
        'correctly_ranked',
        when(col('risk_diff') > tied_tol, 1.0).otherwise(0.0)
    )
    df_valid = df_valid.withColumn(
        'tied',
        when(abs_(col('risk_diff')) <= tied_tol, 0.5).otherwise(0.0)
    )
    
    # Compute numerator and denominator for concordance index
    df_valid = df_valid.withColumn('contribution',
                                   (col('correctly_ranked') + col('tied')) * col('weight_product'))
    
    numerator = df_valid.agg({'contribution': 'sum'}).collect()[0][0]
    denominator = df_valid.agg({'weight_product': 'sum'}).collect()[0][0]
    
    ci = numerator / denominator if denominator != 0 else None
    
    if return_num_valid:
        num_valid = df_valid.count()
        return ci, num_valid
    else:
        return ci


In [ ]:
def xCI(s_test, t_test, pred_risk,
        weights=None,
        pos_group=None, neg_group=None,
        return_num_valid=False,
        tied_tol=1e-8):

    w = weights if weights is not None else np.ones_like(s_test)

    mask1 = (s_test == 1)
    if pos_group is not None:
        mask1 = mask1 & pos_group

    w = w[mask1, np.newaxis]

    mask2 = np.ones_like(s_test, dtype=bool)
    if neg_group is not None:
        mask2 = mask2 & neg_group

    valid = t_test[mask1, np.newaxis] < t_test[np.newaxis, mask2]

    risk_diff = pred_risk[mask1, np.newaxis] - pred_risk[np.newaxis, mask2]

    correctly_ranked = valid & (risk_diff > tied_tol)
    tied = valid & (np.abs(risk_diff) <= tied_tol)

    num_valid = np.sum((w ** 2) * valid)
    ci = np.sum((w ** 2) * (correctly_ranked + 0.5 * tied)) / num_valid

    return (ci, num_valid) if return_num_valid else ci


In [ ]:
s_ascvd = (df['ascvd'] == 1).astype(int).values
t_ascvd = df['fu_ascvd'].values
risk_ascvd = df['ascvd_predicted_probability_10y'].values
female = (df['female'] == 1).astype(int).values
male = (df['female'] == 0).astype(int).values


In [ ]:
for g1, l1 in zip([female, male], ['female', 'male']):

    for g2, l2 in zip([female, male], ['female', 'male']):

        print('xCI %s-%s = %.4f' % (
            l1, l2,
            xCI_spark(
                s_ascvd, t_ascvd, risk_ascvd,
                pos_group=g1, neg_group=g2
            )
        ))

In [ ]:
full_group =[
    "female", "ethnicity", "race", "cat_insurance", "cat_highest_edu_individual",
    "cat_med_nonadherence_risk", "cat_address_own_or_rent",
    "cat_current_address_dwelling_type", "cat_health_mgmt_nonmotivation_risk",
    "cat_highest_edu_household", "cat_healthcare_cost_risk", 
    "cat_address_change_60_months", 
    "cat_med_adherence_rate", "cat_current_address_median_income"
]
 
selected_groups_1 = [
    "female", "ethnicity", "race", "cat_insurance", "cat_highest_edu_individual"]

selected_groups_2 = [
    "cat_address_own_or_rent",
    "cat_current_address_dwelling_type", "cat_current_address_median_income"
]


s_ascvd = (df['ascvd'] == 1).astype(int).values
t_ascvd = df['fu_ascvd'].values
risk_ascvd = df['ascvd_predicted_probability_10y'].values

In [ ]:

for var in selected_groups_1:
    print(f"\n--- xCI for {var} ---")
    levels = df[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (df[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (df[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


In [ ]:

for var in selected_groups_2:
    print(f"\n--- xCI for {var} ---")
    levels = df[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (df[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (df[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


# Calculate xCI for PCE

In [ ]:
pce_df = study.load_artifacts_data("/PREVENT_PCE/Aim_1/original_pce_result")
display_df(pce_df)
pce_df = pce_df.to_pandas()

In [ ]:
pce_s = (pce_df['ascvd'] == 1).astype(int).values
pce_t = pce_df['fu_ascvd'].values
pce_risk = pce_df['ascvd_predicted_probability_10y'].values

In [ ]:
for var in selected_groups_1:
    print(f"\n--- xCI for {var} ---")
    levels = pce_df[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (pce_df[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (pce_df[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    pce_s, pce_t, pce_risk,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


In [ ]:
for var in selected_groups_2:
    print(f"\n--- xCI for {var} ---")
    levels = pce_df[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (pce_df[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (pce_df[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    pce_s, pce_t, pce_risk,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


# Local models 

## Local PREVENT

In [ ]:
local_prevent = study.load_artifacts_data("/PREVENT_PCE/Aim_2_3/prevent_result")
display_df(local_prevent)
local_prevent = local_prevent.to_pandas()

In [ ]:
s_ascvd = (local_prevent['ascvd'] == 1).astype(int).values
t_ascvd = local_prevent['fu_ascvd'].values
risk_ascvd = local_prevent['ascvd_local_10y'].values

In [ ]:
for var in selected_groups_1:
    print(f"\n--- xCI for {var} ---")
    levels = local_prevent[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (local_prevent[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (local_prevent[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


In [ ]:
for var in selected_groups_2:
    print(f"\n--- xCI for {var} ---")
    levels = local_prevent[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (local_prevent[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (local_prevent[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


## Local PCE

In [ ]:
local_pce = study.load_artifacts_data("/PREVENT_PCE/Aim_2_3/pce_result")
display_df(local_pce)
local_pce = local_pce.to_pandas()

In [ ]:
s_ascvd = (local_pce['ascvd'] == 1).astype(int).values
t_ascvd = local_pce['fu_ascvd'].values
risk_ascvd = local_pce['ascvd_local_10y'].values

In [ ]:
for var in selected_groups_1:
    print(f"\n--- xCI for {var} ---")
    levels = local_pce[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (local_pce[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (local_pce[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


In [ ]:
for var in selected_groups_2:
    print(f"\n--- xCI for {var} ---")
    levels = local_pce[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (local_pce[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (local_pce[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


## PREVENT with SDOH

In [ ]:
s_ascvd = (local_prevent['ascvd'] == 1).astype(int).values
t_ascvd = local_prevent['fu_ascvd'].values
risk_ascvd = local_prevent['ascvd_sdoh_10y'].values

In [ ]:
for var in selected_groups_1:
    print(f"\n--- xCI for {var} ---")
    levels = local_prevent[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (local_prevent[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (local_prevent[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


In [ ]:
for var in selected_groups_2:
    print(f"\n--- xCI for {var} ---")
    levels = local_prevent[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (local_prevent[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (local_prevent[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


# PCE with SDOH

In [ ]:
s_ascvd = (local_pce['ascvd'] == 1).astype(int).values
t_ascvd = local_pce['fu_ascvd'].values
risk_ascvd = local_pce['ascvd_sdoh_10y'].values

In [ ]:
for var in selected_groups_1:
    print(f"\n--- xCI for {var} ---")
    levels = local_pce[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (local_pce[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (local_pce[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


In [ ]:
for var in selected_groups_2:
    print(f"\n--- xCI for {var} ---")
    levels = pce_df[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (pce_df[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (pce_df[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    pce_s, pce_t, pce_risk,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


# DNN

In [ ]:
dnn = study.load_artifacts_data("/data/DNN_result/ascvd_result")
display_df(dnn)
dnn = dnn.to_pandas()

In [ ]:
s_ascvd = (dnn['ascvd'] == 1).astype(int).values
t_ascvd = dnn['fu_ascvd'].values
risk_ascvd = dnn['cvd_risk_year_10'].values

In [ ]:
for var in selected_groups_1:
    print(f"\n--- xCI for {var} ---")
    levels = dnn[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (dnn[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (dnn[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


In [ ]:
for var in selected_groups_2:
    print(f"\n--- xCI for {var} ---")
    levels = dnn[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (dnn[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (dnn[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    s_ascvd, t_ascvd, risk_ascvd,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")
